# AarogyaAI — Exploratory Data Analysis

General-purpose exploration notebook. Run cells in order after placing data in `data/raw/`.

In [ ]:
import sys
from pathlib import Path

# Always resolve to project root regardless of notebook CWD
ROOT = Path('__file__').resolve().parent if '__file__' in dir() else Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

print('Project root:', ROOT)
print('Python path set.')

In [ ]:
# ── Dataset availability check ──────────────────────────────────────────────
datasets = {
    'Yoga-82 train':        ROOT / 'data' / 'raw' / 'yoga82' / 'yoga_train',
    'Yoga-82 test':         ROOT / 'data' / 'raw' / 'yoga82' / 'yoga_test',
    'BlazePose CSV':        ROOT / 'data' / 'raw' / 'blazepose_yoga82' / 'keypoints_train.csv',
    'Yoga Poses 5-class':   ROOT / 'data' / 'raw' / 'yoga_poses_5class' / 'DATASET' / 'TRAIN',
}
for name, path in datasets.items():
    status = '✅ FOUND' if path.exists() else '❌ MISSING'
    print(f'{status}: {name}')

In [ ]:
# ── Pose metadata overview ────────────────────────────────────────────────────
from ml.recommendation.yoga_metadata import poses_to_dataframe

df_poses = poses_to_dataframe()
print(f'Pose catalog: {len(df_poses)} entries')
print(df_poses.groupby(['category', 'difficulty']).size().unstack(fill_value=0))

In [ ]:
# ── Feature engineering overview ─────────────────────────────────────────────
from ml.preprocessing.skeleton_utils import SKELETON_FEATURE_DIM
from ml.recommendation.engine import POSE_VECTOR_DIM
from ml.features import FEATURE_NAMES as PROFILE_FEATURES

print(f'Profile feature vector dim   : {len(PROFILE_FEATURES)} — {PROFILE_FEATURES}')
print(f'Skeleton feature vector dim  : {SKELETON_FEATURE_DIM}')
print(f'Pose/user vector dim (engine): {POSE_VECTOR_DIM}')

In [ ]:
# ── Synthetic profile distribution (available immediately) ───────────────────
from ml.models.user_goal_classifier import generate_synthetic_profiles
import matplotlib.pyplot as plt

df_synth = generate_synthetic_profiles(n_samples=500)
print('⚠️  SYNTHETIC DATA — for pipeline demonstration only')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_synth['goal_label'].value_counts().plot(kind='bar', ax=axes[0], title='Goal Distribution')
df_synth['bmi'].hist(bins=30, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('BMI Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# ── Processed Yoga-82 stats (after preprocessing) ───────────────────────────
import json
stats_path = ROOT / 'data' / 'processed' / 'metadata' / 'dataset_stats.json'
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    print(f"Classes     : {stats['n_classes']}")
    print(f"Corrupted   : {len(stats['corrupted'])}")
    print(f"Duplicates  : {stats['duplicates_removed']}")
    # Per-class image counts
    counts = {k: v['valid'] for k, v in stats['per_class'].items()}
    df_counts = pd.Series(counts).sort_values()
    print(f"Min imgs/class: {df_counts.min()}, Max: {df_counts.max()}, Mean: {df_counts.mean():.1f}")
else:
    print('dataset_stats.json not found — run scripts/preprocess.py --yoga82 after downloading Yoga-82')